<a id="unit-u.115351f82c464f66a41c8d091a466d7f"></a>

<div>
  <div>3-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
  <div>Модель семантикасы, бөлімдеу және бакеттерге бөлу</div>
</div>

Бірдей үш жолды Duplicate, Unique және Aggregate Key кестелеріне жазып, әрқайсысы не сақтайтынын салыстырыңыз. Содан кейін он тапсырысты күн бойынша бөлімдеп, тапсырыс ID-і бойынша бакеттерге бөліңіз және үш жоспардан күн сүзгісі мен тапсырыс ID-інің сүзгісі сұрау сканерлейтін ауқымды қалай тарылтатынын оқыңыз. Шамамен 25 минут жоспарлаңыз.

**3-модуль: Doris кесте модельдері, бөлімдеу және бакеттерге бөлу** · [Сабақ](course.md) · [Тест](quiz03_models_and_data_distribution.ipynb) · [Курстың README файлы](../../README.md)


<a id="unit-u.ded81b50879b4115b80d31b35613555c"></a>

<a id="lab-scope"></a>

## Зертханалық жұмыстың қамту аясы

Бұл жұмыс тек `orders_duplicate`, `orders_unique`, `orders_aggregate` және `orders_partitioned` кестелерін қайта жасайды; жаттығулар `orders_model_practice` кестесін қайта жасайды. 1-зертханалық жұмыстың курс сынақ ортасы қажет.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows
from dw_course.ui import show_sql, show_response

lab = connect_sandbox()




<a id="unit-u.0fbd1d0a06e947288800a10c15b98d34"></a>

<a id="1-same-keys-different-models"></a>

## 1. Бірдей кілттер, әртүрлі модельдер

WWI үлгісінен 1-тапсырысты (2300.00) және 2-тапсырысты (405.00) алып, зертханалық кестелерде 1-тапсырысты 2250.00-ге түзетуді модельдеңіз. Үш жіберуді ретімен аяқтағаннан кейін бақылаңыз:

- Duplicate Key үш кіріс жазбасының бәрін сақтайды.
- Unique Key екі тапсырысты сақтайды, 1-тапсырыстың сомасы — 2250.00.
- Aggregate Key кілт бойынша қосып, 1-тапсырыс үшін 4550.00 береді.

Соңғы нәтиже қосылатын метрикаларға жарайды; түзетуден кейінгі тапсырыстың ағымдағы сомасын көрсету үшін кілт бойынша жаңарту семантикасын қолданыңыз.

Unique Key кестесінде `"enable_unique_key_merge_on_write" = "true"` орнатылған: **Merge-on-Write** (MoW) бірдей кілтті жолдарды жазу кезінде шешеді, сондықтан сұрау тек соңғысын оқиды. Doris 4.x нұсқасында бұл Unique Key кестелерінің әдепкі тәсілі; 7-модуль оның нұсқалармен және ішінара жаңартулармен жұмысын көрсетеді.


In [ ]:
definitions = {
    "orders_duplicate": 'CREATE TABLE orders_duplicate (id BIGINT, amount DECIMAL(12,2)) DUPLICATE KEY(id) DISTRIBUTED BY HASH(id) BUCKETS 1 PROPERTIES("replication_num"="1")',
    "orders_unique": 'CREATE TABLE orders_unique (id BIGINT, amount DECIMAL(12,2)) UNIQUE KEY(id) DISTRIBUTED BY HASH(id) BUCKETS 1 PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")',
    "orders_aggregate": 'CREATE TABLE orders_aggregate (id BIGINT, amount DECIMAL(12,2) SUM) AGGREGATE KEY(id) DISTRIBUTED BY HASH(id) BUCKETS 1 PROPERTIES("replication_num"="1")',
}
for table, ddl in definitions.items():
    lab.execute(f"DROP TABLE IF EXISTS {table}")
    show_sql("CREATE TABLE SQL", ddl)
    lab.execute(ddl)
    for row in [(1, "2300.00"), (1, "2250.00"), (2, "405.00")]:
        lab.insert(table, ["id", "amount"], [row])
expect(lab.query("SELECT id, amount FROM orders_duplicate ORDER BY id, amount"), [(1,"2250.00"),(1,"2300.00"),(2,"405.00")],
       title="Duplicate Key keeps all three rows")
expect(lab.query("SELECT id, amount FROM orders_unique ORDER BY id"), [(1,"2250.00"),(2,"405.00")],
       title="Unique Key keeps the later amount of order 1")
expect(lab.query("SELECT id, amount FROM orders_aggregate ORDER BY id"), [(1,"4550.00"),(2,"405.00")],
       title="Aggregate Key sums the two amounts of order 1")
for table in definitions:
    lab.sql(f"SELECT * FROM {table} ORDER BY id, amount", title=table + ": Results for the same input")

<a id="unit-u.ebfe2287c785421c80524629367f42a5"></a>

**Күтілетін нәтиже**

Үш кесте анықтамасы, үш жасыл белгі, содан кейін үш кесте:

| Кесте | Жолдар |
| --- | --- |
| orders_duplicate | (1, 2250.00), (1, 2300.00), (2, 405.00) |
| orders_unique | (1, 2250.00), (2, 405.00) |
| orders_aggregate | (1, 4550.00), (2, 405.00) |

Бірдей үш жазу үш жауап береді: әр жол, әр кілттің соңғы жолы және әр кілттің қосындысы.


<a id="unit-u.3493ec419cef4af7a3b4188e073d93b5"></a>

<a id="2-what-partitioning-and-bucketing-each-control"></a>

## 2. Бөлімдеу мен бакеттерге бөлу нені басқарады?

Бөлімделген кесте бастапқы он тапсырысты қолданады: әр күнге бір бөлім, әр бөлім тапсырыс ID-інің хэші бойынша төрт бакетке бөлінеді. Толық кесте сұрауының, тек күн сүзгісінің және күн мен тапсырыс ID-і біріктірілген сүзгінің жоспарларын ретімен салыстырып, таңдалған бөлім мен Tablet ауқымдарының қалай тарылатынын бақылаңыз.

Бұл кесте тарихи егжей-тегжейлі жазбаларды сақтайды, күн мен тапсырыс ID-і сұрыптау үшін қолданылады; ағымдағы тапсырыс кестесінің бизнестегі бірегей кілтін бөлек жобалау керек.


In [ ]:
from dw_course.wwi import sample
lab.execute("DROP TABLE IF EXISTS orders_partitioned")
lab.execute("""
CREATE TABLE orders_partitioned (
    order_date DATE, order_id BIGINT, amount DECIMAL(18,2)
) DUPLICATE KEY(order_date, order_id)
PARTITION BY RANGE(order_date) (
    PARTITION p_day1 VALUES [('2013-01-01'), ('2013-01-02')),
    PARTITION p_day2 VALUES [('2013-01-02'), ('2013-01-03'))
)
DISTRIBUTED BY HASH(order_id) BUCKETS 4
PROPERTIES("replication_num"="1")
""")
lab.insert("orders_partitioned", ["order_date", "order_id", "amount"],
           [(r["order_date"], r["order_id"], r["order_amount"]) for r in sample()["orders"]])
for query in [
    "SELECT * FROM orders_partitioned",
    "SELECT * FROM orders_partitioned WHERE order_date = '2013-01-01'",
    "SELECT * FROM orders_partitioned WHERE order_date = '2013-01-01' AND order_id = 1",
]:
    lab.sql("EXPLAIN " + query)
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM orders_partitioned WHERE order_date = '2013-01-01'"),
       [(5, "3944.20")], title="Pruning does not change the answer: five orders, 3944.20 on the first day")

<a id="unit-u.2e666b69b2da4a7b84f41a90089ed961"></a>

**Күтілетін нәтиже**

Үш жоспар. Әрқайсысынан `VOlapScanNode` сканерлеу түйінін және оның `partitions=` мен `tablets=` жолдарын табыңыз:

| Сұрау | partitions= | tablets= |
| --- | --- | --- |
| Сүзгісіз | `2/2 (p_day1,p_day2)` | `8/8` |
| `order_date = '2013-01-01'` | `1/2 (p_day1)` | `4/4` |
| `order_date = '2013-01-01' AND order_id = 1` | `1/2 (p_day1)` | `1/4` |

Күн сүзгісі екі бөлімнің бірін қалдырады; содан кейін `tablets=` қалған бөлімдердің tablet-терін санайды, олардың саны — 4. Бакеттерге бөлу бағаны — тапсырыс ID-і — tablet-інде 1-тапсырыс болуы мүмкін жалғыз бакетті таңдайды. Сүзгілер `PREDICATES` ретінде де көрінеді. Содан кейін жасыл белгі шығады: бірінші күнде әлі бес тапсырыс және 3944.20 бар. Жоспар сұраудың қанша уақыт алатынын емес, не оқитынын көрсетеді.


<a id="unit-u.6269cc12ac5f422fbef65bbd2589d242"></a>

<a id="3-practice-which-result-can-a-report-use"></a>

## 3. Жаттығу: есеп қай нәтижені қолдана алады?

Есепке 1-тапсырыстың түзетуден кейінгі ағымдағы сомасы қажет. Үш модель кестесінде де 1-тапсырысты және `orders_partitioned` кестесінде бірінші күннің тапсырыстары мен сомасын сұраңыз. Есептің сұрағына қай кесте жауап береді және 4550.00 неліктен 1-тапсырыста ешқашан болмаған сома?


In [ ]:
# Your code here

<a id="unit-u.8c21960832cd4fef971787362c23fe6b"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
lab.sql("""SELECT 'duplicate' AS model, COUNT(*) AS rows_for_order_1, SUM(amount) AS summed FROM orders_duplicate WHERE id = 1
UNION ALL SELECT 'unique', COUNT(*), SUM(amount) FROM orders_unique WHERE id = 1
UNION ALL SELECT 'aggregate', COUNT(*), SUM(amount) FROM orders_aggregate WHERE id = 1
ORDER BY model""", title="Order 1 in the three models");
expect(lab.query("SELECT amount FROM orders_unique WHERE id = 1"), [("2250.00",)],
       title="Only Unique Key gives the corrected amount")
expect(lab.query("SELECT COUNT(*), SUM(amount) FROM orders_partitioned WHERE order_date = '2013-01-01'"),
       [(5, "3944.20")], title="The first day in orders_partitioned: five orders, 3944.20")

<a id="unit-u.50e021be7b4b4209b9256530dc21761f"></a>

**Күтілетін нәтиже**

Үш жол: 1 жолы және 4550.00 сомасы бар `aggregate`, жалпы сомасы 4550.00 болатын 2 жолы бар `duplicate` және 2250.00 сомалы 1 жолы бар `unique`, содан кейін екі жасыл белгі. Тек Unique Key кестесінде түзетілген 2250.00 сомасы сақталған. Aggregate кестесі бастапқы және түзетілген сомаларды қосты; бұл екі сату сияқты қосылатын шамалар үшін дұрыс, ал бір соманың екі нұсқасы үшін қате. Бөлімделген кестеде 1-тапсырыстың бастапқы 2300.00 сомасы әлі бар: онда он бастапқы тапсырыс сақталған, бесеуі бірінші күнде, сомасы 3944.20.


<a id="unit-u.c60e720e151a4a0e8e27b7ce31e52762"></a>

<a id="4-practice-choose-a-model-for-the-current-amount"></a>

## 4. Жаттығу: ағымдағы сома үшін модель таңдау

Бизнеске тапсырыстардың ағымдағы сомаларын сұрау қажет. (501,100), (501,80) және (502,40) мәндерін ретімен қабылдаңыз, әр жазуды алдыңғысы аяқталғаннан кейін ғана жіберіңіз. Бөлек orders_model_practice кестесі үшін модель таңдап, оны жасаңыз және деректерді жазыңыз; 501-тапсырыс үшін 80, 502-тапсырыс үшін 40 күтіледі. Егжей-тегжейлі деректер моделі мен SUM агрегаттауы қандай басқа нәтижелер беретінін ойлаңыз.

Бұл жаттығу тек orders_model_practice кестесін қайта жасайды.

Кодты келесі ұяшыққа жазыңыз, содан кейін одан кейінгі үлгі шешіммен салыстырыңыз.


In [ ]:
# Your code here

<a id="unit-u.acfc3108ac914f3189418ff7048877bd"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.


In [ ]:
lab.execute("DROP TABLE IF EXISTS orders_model_practice")
lab.execute("""CREATE TABLE orders_model_practice (order_id BIGINT, amount DECIMAL(12,2))
UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")""")
lab.execute("INSERT INTO orders_model_practice VALUES (501, 100)")
lab.execute("INSERT INTO orders_model_practice VALUES (501, 80)")
lab.execute("INSERT INTO orders_model_practice VALUES (502, 40)")
lab.sql("SELECT * FROM orders_model_practice ORDER BY order_id", title="Current amounts of the two orders");
expect(lab.query("SELECT * FROM orders_model_practice ORDER BY order_id"), [(501,"80.00"),(502,"40.00")],
       title="Order 501 is 80.00 and order 502 is 40.00")

<a id="unit-u.60c761f0ab7e446e942fd25693480704"></a>

**Күтілетін нәтиже**

Екі жол: 80.00 сомасы бар 501 және 40.00 сомасы бар 502, жасыл белгі. Unique Key кестесі 501-тапсырыстың кейінгі сомасын сақтайды; Duplicate Key кестесі 501-дің екі жолын да сақтар еді, ал `SUM` қолданатын Aggregate кестесі 180.00 көрсетер еді.


<a id="unit-u.408bfa31d65643f0b931896ff5193ece"></a>

<a id="you-are-done-when"></a>

## Жұмысты аяқтау шарттары

- бірдей жазулардан Duplicate, Unique және Aggregate Key кестелері не сақтайтынын және «қазіргі сома қандай?» сұрағына қайсысы жауап беретінін болжай аласыз;
- жоспардағы `partitions=` және `tablets=` өрістерін оқып, қай сүзгі қайсысын тарылтқанын айта аласыз;
- әр тексеруде бірінші күнде әлі бес тапсырыс және 3944.20 бар: артық деректерді оқудан алып тастау жауапты емес, оқылған деректерді өзгертті.


<a id="unit-u.e91c261965bc49b1a61b689f7dede9c6"></a>

<a id="next"></a>

## Келесі қадам

[3-тест: Doris кесте модельдері, бөлімдеу және бакеттерге бөлу](quiz03_models_and_data_distribution.ipynb) файлын ашыңыз, содан кейін [4-модуль: сыртқы деректерді сұрау және деректер көлі кестелерін біріктіру](../module04-external-access/course.md) бөліміне өтіңіз.
